# MapReduce-Reloaded: the demo, end to end

This notebook runs the demo in `docs/runbook.md` §5 against the Compose
cluster: upload the corpus, submit word count, read the output, plot it,
submit the same program as an interpreted job file, run a second program,
kill a worker in the middle of a job, look at the metrics, and check local
mode and the cluster against the checked-in expected output.

**Before you start**, on the VM, from the repository root:

```sh
make cluster-up mr      # the cluster and the client
make upload-example     # the corpus and a 10 MB web log
python/.venv/bin/pip install -e 'python[notebook]'   # Jupyter and matplotlib
```

Then open it with `python/.venv/bin/jupyter notebook examples/demo.ipynb`, or
run every cell without a browser:

```sh
python/.venv/bin/jupyter nbconvert --to notebook --execute --inplace examples/demo.ipynb
```

*Kernel → Restart & Run All* takes about a minute on the course VM: about 5
seconds without section 9, the worker kill, which sorts 1 GiB (it generates
the input on first use, about 20 seconds and 1 GB of disk). Set
`KILL_WORKER = False` in the setup cell to skip it. Rerunning the notebook
from the top is safe: each run uses timestamped output directories, and the
last cell removes them. The status page is at <http://127.0.0.1:18780/>.


## 1. Setup


In [ ]:
import os
import shutil
import subprocess
import time
import urllib.error
import urllib.request
from pathlib import Path

# Section 9 kills a worker during a 1 GiB sort; False skips it.
KILL_WORKER = True


def find_root():
    """The repository root: MR_ROOT, or the first of the kernel's working
    directory and its parents that holds deploy/compose. Jupyter starts the
    kernel in examples/, the notebook's directory."""
    if "MR_ROOT" in os.environ:
        return Path(os.environ["MR_ROOT"])
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / "deploy" / "compose" / "compose.yaml").exists():
            return path
    raise RuntimeError("not inside the repository: set MR_ROOT to its path")


ROOT = find_root()
MR = ROOT / "bin" / "mr"
RUN = f"{int(time.time())}-{os.getpid()}"  # unique per run, even within a second
CORPUS = f"/data/input/notebook-corpus-{RUN}"
COMPOSE = ["docker", "compose", "-p", "mapreduce", "-f", "deploy/compose/compose.yaml"]
METRICS = "http://127.0.0.1:18780/metrics"
EXPECTED = b"".join(p.read_bytes() for p in sorted((ROOT / "testdata/corpus/expected").glob("part-r-*")))
JOBS = []  # this run's job ids, for section 10


def run(*args, check=True):
    """Runs a command in the repository root and returns the result."""
    result = subprocess.run([str(a) for a in args], cwd=ROOT, text=True, capture_output=True)
    if check and result.returncode != 0:
        raise RuntimeError(f"{' '.join(map(str, args))} failed:\n{result.stderr}")
    return result


def submit(*args):
    """Submits a job with -wait, records its id and returns its status reply."""
    reply = run(MR, "submit", "-wait", *args).stdout
    job, state = reply.splitlines()[0].split("\t")
    assert state == "SUCCEEDED", reply
    JOBS.append(job)
    return reply


def cat(path):
    """A job's output, its part files in order, as bytes."""
    return subprocess.run([MR, "cat", path], cwd=ROOT, capture_output=True, check=True).stdout


def metrics_lines():
    """The coordinator's /metrics, as text lines."""
    with urllib.request.urlopen(METRICS, timeout=5) as response:
        return response.read().decode().splitlines()


def fields(text):
    """A status reply, as a dict: job-1\tSUCCEEDED, map\t4/4, counters."""
    return dict(line.split("\t", 1) for line in text.splitlines() if "\t" in line)


assert MR.exists(), "bin/mr is missing: run `make cluster-up mr` first"
try:
    metrics_lines()
except (urllib.error.URLError, OSError) as error:
    raise RuntimeError(f"the coordinator is not answering at {METRICS}: run `make cluster-up`") from error
# A job waits forever for a worker, so stop here if none is live.
workers = next((int(line.split()[1]) for line in metrics_lines() if line.startswith("mr_workers ")), 0)
if workers == 0:
    raise RuntimeError("the coordinator has no live workers: run `make cluster-up`")
print(f"repository: {ROOT}, live workers: {workers}")


## 2. Check the cluster

One coordinator and four workers, each limited to one CPU.


In [ ]:
ps = run(*COMPOSE, "ps", "--format", "{{.Service}}\t{{.Status}}")
print(ps.stdout or "no containers: run `make cluster-up mr`")
print(next((line for line in metrics_lines() if line.startswith("mr_workers ")), "no mr_workers metric"))


## 3. Upload the corpus

Every container sees the host's `data/` directory at `/data`.


In [ ]:
run(MR, "upload", "testdata/corpus/*.txt", CORPUS)
print(sorted(p.name for p in (ROOT / CORPUS.removeprefix("/")).iterdir()))


## 4. Submit word count

The compiled `wordcount` program, by name: 4 map tasks and 4 reduce tasks.


In [ ]:
WC = f"/data/out/notebook-wc-{RUN}"
reply = submit("-m", "4", "-r", "4", "wordcount", WC, f"{CORPUS}/*.txt")
print(reply)
status = fields(reply)
print({key: status[key] for key in ("map", "reduce", "seconds")})


## 5. Read the output


In [ ]:
output = cat(WC)
words = [line.split("\t") for line in output.decode().splitlines()]
top = sorted(((word, int(count)) for word, count in words), key=lambda pair: pair[1], reverse=True)[:10]
print(f"{len(words):,} distinct words")
for word, count in top:
    print(f"{word:>16} {count:>8,}")


## 6. Plot the ten most frequent words


In [ ]:
import matplotlib.pyplot as plt

figure, axis = plt.subplots(figsize=(7, 3.5))
axis.barh([word for word, _ in reversed(top)], [count for _, count in reversed(top)], color="#3b6ea5")
axis.set_xlabel("occurrences")
axis.set_title("Ten most frequent words, cluster word count")
figure.tight_layout()
plt.show()


## 7. The same job as a job file

`examples/wordcount.go` is word count as Go source. It is sent with the job
and interpreted by the workers (ADR 0015): nothing is compiled or uploaded.
Its output must be the compiled program's, byte for byte.


In [ ]:
JOBFILE = f"/data/out/notebook-jobfile-{RUN}"
reply = submit("-m", "4", "-r", "4", "examples/wordcount.go", JOBFILE, f"{CORPUS}/*.txt")
print({key: value for key, value in fields(reply).items() if key in ("map", "reduce", "seconds")})
assert cat(JOBFILE) == output, "the job file and the compiled program differ"
print("job file and compiled program agree: True")


## 8. A second program: the most requested pages


In [ ]:
assert (ROOT / "data/input/weblog/access.log").exists(), "run `make upload-example` first"
URLS = f"/data/out/notebook-urls-{RUN}"
submit("-m", "2", "urlfreq", URLS, "/data/input/weblog/access.log")
urls = [line.split("\t") for line in cat(URLS).decode().splitlines()]
for url, count in sorted(((url, int(n)) for url, n in urls), key=lambda pair: pair[1], reverse=True)[:5]:
    print(f"{count:>8,}  {url}")


## 9. Kill a worker in the middle of a job

The coordinator notices a dead worker after a 10-second heartbeat timeout,
so this uses a job that lasts longer: a 1 GiB sort with 64 map tasks. Five
seconds in, `docker kill` stops worker 4. The job still ends `SUCCEEDED`; the
coordinator's log says how many of the dead worker's tasks ran again
elsewhere (§3.3 of the paper). The cell then restarts the worker.


In [ ]:
SORT_INPUT = ROOT / "data/input/sort-1g.txt"
SORT = f"/data/out/notebook-sort-kill-{RUN}"
if not KILL_WORKER:
    print("skipped: KILL_WORKER is False")
else:
    if not SORT_INPUT.exists():
        free = shutil.disk_usage(ROOT / "data").free
        assert free > 3 * 2**30, f"{free / 2**30:.1f} GiB free: the sort needs 3 GiB"
        with SORT_INPUT.open("wb") as out:
            subprocess.run(["python3", "bench/gen-sort.py", str(2**30), "42"], cwd=ROOT, stdout=out, check=True)
    job = subprocess.Popen(
        [MR, "submit", "-m", "64", "-r", "4", "-wait", "distsort", SORT, "/data/input/sort-1g.txt"],
        cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    )
    try:
        time.sleep(5)
        run("docker", "kill", "mapreduce-worker-4")
        print("killed mapreduce-worker-4 five seconds into the job")
        reply, _ = job.communicate()
    finally:
        run(*COMPOSE, "up", "-d", "--no-build", "worker")
        for _ in range(30):  # until the restarted worker registers
            if "mr_workers 4" in metrics_lines():
                break
            time.sleep(1)
        else:
            print("warning: the restarted worker has not registered after 30 s; check `docker compose ps`")
    status = fields(reply)
    JOBS.append(reply.split("\t")[0])
    print(reply.splitlines()[0], {key: status[key] for key in ("map", "reduce", "seconds")})
    assert reply.splitlines()[0].endswith("SUCCEEDED"), reply
    assert status["reduce.output.records"] == status["map.input.records"], "records lost"
    logs = run("docker", "logs", "mapreduce-coordinator-1")
    log = (logs.stdout + logs.stderr).splitlines()
    print(next((line for line in reversed(log) if "worker lost" in line), "no worker lost line"))


## 10. The coordinator's metrics

Completed tasks for this run's jobs.


In [ ]:
for line in metrics_lines():
    if line.startswith("mr_workers ") or ('state="completed"' in line and any(f'job="{job}"' in line for job in JOBS)):
        print(line)


## 11. Local mode is the oracle

Local mode runs the same job in one process, without Docker. Its output, the
cluster's and the expected output checked in at `testdata/corpus/expected`
must all be the same, byte for byte (README, "Run a job on one machine").


In [ ]:
LOCAL = f"data/out/notebook-local-{RUN}"
run(MR, "run", "-local", "-m", "4", "-r", "4", "wordcount", LOCAL, "testdata/corpus/*.txt")
assert cat(f"/{LOCAL}") == output, "local mode and the cluster differ"
print("local mode and the cluster agree: True")
assert output == EXPECTED, "the cluster's output differs from testdata/corpus/expected"
print("the cluster matches testdata/corpus/expected: True")


## 12. Clean up

This run's input and output. The sort input stays in `data/input/` for the next run.


In [ ]:
for path in (WC, JOBFILE, URLS, SORT, f"/{LOCAL}", CORPUS):
    shutil.rmtree(ROOT / path.removeprefix("/"), ignore_errors=True)
print("removed this run's input and output")
